# BugBuster Desktop App

Builds and launches the **Tauri + Leptos** desktop application.

| Cell | Action |
|------|--------|
| Setup | Verify toolchain and paths |
| Dev mode | `cargo tauri dev` — hot-reload, opens window automatically |
| Release build | `cargo tauri build` — produces installer/exe |
| Launch | Run the already-built executable directly |

In [ ]:
import subprocess
import os
import sys
import threading
from pathlib import Path
from IPython.display import display, HTML

def check_tool(cmd, args=["--version"]):
    try:
        r = subprocess.run([cmd] + args, capture_output=True, text=True)
        return r.stdout.strip() or r.stderr.strip()
    except FileNotFoundError:
        return None

def find_tool(name, args=["--version"]):
    """Check PATH first, then ~/.cargo/bin/ fallback."""
    return check_tool(name, args) or check_tool(str(Path.home() / ".cargo" / "bin" / name), args)

# Auto-detect app directory relative to this notebook.
# Works on any OS: notebook is at <repo>/notebooks/, app at <repo>/DesktopApp/BugBuster/
_notebook_dir = Path.cwd()
_candidates = [
    _notebook_dir / ".." / "DesktopApp" / "BugBuster",
    _notebook_dir / "DesktopApp" / "BugBuster",
    _notebook_dir.parent / "DesktopApp" / "BugBuster",
]
if sys.platform == "win32":
    _candidates.append(Path(r"C:\Users\Public\Documents\Altium\BugBuster\DesktopApp\BugBuster"))
elif sys.platform == "darwin":
    _candidates.append(Path.home() / "Desktop" / "BugBuster" / "DesktopApp" / "BugBuster")

APP_DIR = None
for c in _candidates:
    if c is not None:
        resolved = c.resolve()
        if resolved.exists() and (resolved / "Cargo.toml").exists():
            APP_DIR = resolved
            break

if APP_DIR is None:
    print("⚠ Could not find app directory automatically.")
    print("  Set it manually: APP_DIR = Path('/path/to/DesktopApp/BugBuster')")
    APP_DIR = Path(".")

# Find cargo binary (may not be on PATH on macOS)
CARGO = "cargo"
if not check_tool("cargo"):
    _cargo_home = Path.home() / ".cargo" / "bin" / "cargo"
    if _cargo_home.exists():
        CARGO = str(_cargo_home)

tools = {
    "rustc":       find_tool("rustc"),
    "cargo":       find_tool("cargo"),
    "trunk":       find_tool("trunk"),
    "cargo-tauri": find_tool("cargo", ["tauri", "--version"]),
}

print("=== Toolchain check ===")
all_ok = True
for name, ver in tools.items():
    status = "OK" if ver else "MISSING"
    print(f"  {name:<15} {status}  {ver or ''}")
    if not ver:
        all_ok = False

print()
print(f"App directory : {APP_DIR}")
print(f"Directory OK  : {APP_DIR.exists()}")
print(f"Cargo binary  : {CARGO}")
if not all_ok:
    print("\n⚠ Some tools are missing. Install them before running.")

In [ ]:
# ─── Stream subprocess output live ───────────────────────────────────────────
import signal

_active_proc = None

def run_streaming(cmd, cwd=APP_DIR, env=None):
    """Run a command and stream stdout/stderr to the notebook in real-time."""
    global _active_proc
    merged_env = {**os.environ, **(env or {})}
    print(f"$ {' '.join(cmd)}")
    print("-" * 60)
    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=merged_env,
    )
    _active_proc = proc
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
    except KeyboardInterrupt:
        proc.terminate()
        print("\n[Interrupted — process terminated]")
    finally:
        proc.wait()
        _active_proc = None
    print("-" * 60)
    print(f"Exited with code {proc.returncode}")
    return proc.returncode

def stop_app():
    """Terminate the running app process (call from a new cell if needed)."""
    global _active_proc
    if _active_proc and _active_proc.poll() is None:
        _active_proc.terminate()
        print("Process terminated.")
    else:
        print("No active process.")

print("Helpers loaded. Use stop_app() in a new cell to terminate a running process.")

## Development mode
Starts `trunk serve` (frontend) + Tauri backend simultaneously.  
The app window opens automatically. **This cell runs until you stop it** (Kernel → Interrupt or call `stop_app()`).

In [ ]:
run_streaming([CARGO, "tauri", "dev"])

## Release build
Compiles an optimised binary and bundles an installer under `src-tauri/target/release/bundle/`.

In [ ]:
run_streaming([CARGO, "tauri", "build"])

## Launch already-built executable
Finds and launches the `.exe` produced by the release build without rebuilding.

In [ ]:
import glob

# Search for the built executable (cross-platform)
release_dir = APP_DIR / "src-tauri" / "target" / "release"
bundle_dir = release_dir / "bundle"

candidates = []

if sys.platform == "win32":
    candidates.extend(glob.glob(str(release_dir / "*.exe")))
    candidates.extend(glob.glob(str(bundle_dir / "nsis" / "*.exe")))
elif sys.platform == "darwin":
    candidates.extend(glob.glob(str(bundle_dir / "macos" / "*.app")))
    candidates.extend(glob.glob(str(release_dir / "bugbuster*")))
else:  # Linux
    candidates.extend(glob.glob(str(bundle_dir / "appimage" / "*.AppImage")))
    candidates.extend(glob.glob(str(bundle_dir / "deb" / "*.deb")))
    candidates.extend(glob.glob(str(release_dir / "bugbuster*")))

# Filter: prefer the raw binary, skip installers/debug symbols
exes = [p for p in candidates
        if "bugbuster" in Path(p).name.lower()
        and not p.endswith((".msi", ".dSYM", ".d", ".pdb"))]

if not exes:
    print("No built executable found. Run the 'Release build' cell first.")
    print(f"Searched in: {release_dir}")
else:
    exe = exes[0]
    print(f"Launching: {exe}")
    if sys.platform == "darwin" and exe.endswith(".app"):
        subprocess.Popen(["open", exe])
    else:
        subprocess.Popen([exe], cwd=str(APP_DIR))
    print("App started (detached — window will open shortly).")

## Stop the running process
Run this cell to terminate `cargo tauri dev` if still running.

In [ ]:
stop_app()